# How-to use JAX with `qiskit-dynamics`


Integration of `qiskit-dynamics` with JAX enables just-in-time compilation, automatic differentation, and GPU execution. Most parts of `qiskit-dynamics` are written to function internally either using `numpy` or `jax.numpy`, but it's necessary to configure the package to use JAX. 

This how-to guide dmonstrates: 
1. How-to configure `qiskit-dynamics` to work with JAX behind the scenes. 
2. How-to write, and use JAX to compile, a parameterized simulation in `qiskit-dynamics`.

## 1. Configure `qiskit-dynamics` to work with JAX

Configuring `qiskit-dynamics` to work with JAX requires working with the `dispatch`. Most functions in `qiskit-dynamics` can operate in either a `numpy`/`scipy`-mode, or a `jax.numpy`/`jax.scipy` mode, and this is enabled via `qiskit_dynamics.dispatch` and the `Array` class.

In [1]:
import numpy as np
import jax

jax.config.update("jax_enable_x64", True)

from qiskit_dynamics import dispatch
from qiskit_dynamics.dispatch import Array

The `Array` class can wrap different ndarray backends.

In [2]:
a1 = Array([[1., 2.], [3., 4.]], backend='numpy')
print(a1.backend)
print(a1)

numpy
[[1. 2.]
 [3. 4.]]


In [3]:
a2 = Array([[1., 2.], [3., 4.]], backend='jax')
print(a2.backend)
print(a2)

jax
[[1. 2.]
 [3. 4.]]


Calling a `numpy` or `scipy` function on an `Array` will dispatch to use the correct backend. I.e. the following results in a `numpy` array, generated via `numpy.cos`.

In [4]:
np.cos(a1)

Array([[ 0.54030231, -0.41614684],
       [-0.9899925 , -0.65364362]], backend='numpy')

While calling `np.cos` on an `Array` with `backend='jax'` will be executed via `jax.numpy.cos`.

In [5]:
np.cos(a2)

Array([[ 0.54030231, -0.41614684],
       [-0.9899925 , -0.65364362]], backend='jax')

By default, constructing an `Array` will result in an `Array` with `numpy` backend, but the default can be changed to `jax`.

In [6]:
dispatch.set_default_backend('jax')

Now `Array`s will automatically be constructed with `jax` backend.

In [7]:
Array(np.arange(3)).backend

'jax'

## 2. Writing JAX-transformable functions in `qiskit-dynamics`

JAX-transformable functions must be pure, in the sense that they have no side-effects. As `qiskit-dynamics` contains some stateful objects, if the computation of a function involves updating these objects, it is necessary to create *copies* within the function before updating it.

An anticipated common use-case is defining the operator components of a model, and then defining and transforming a function whose inputs are the parameters to the signal coefficients, and the outputs are the result of a simulation. We demonstrate here how to write such a function.

In [8]:
from qiskit.quantum_info import Operator
from qiskit_dynamics import Solver
from qiskit_dynamics.signals import DiscreteSignal

First, instantiate the `Solver` object *without* signals. These will be inserted in the function to be transformed.

In [9]:
drift = 2 * np.pi * 5 * Operator.from_label('Z') / 2

ham_solver = Solver(hamiltonian_operators=[2 * np.pi * 0.02 * Operator.from_label('X') / 2],
                    drift=drift,
                    rotating_frame=drift)

Next, define the function. We will consider driving with `DiscreteSignal` on resonance. The input to the function will be the samples, and the output will be the unitary resulting from the time-evolution under the signal.

In [10]:
dt = 1.

def simulation_function(samples):
    
    # define signal
    signal = DiscreteSignal(samples=samples, dt=dt, carrier_freq=5.)
    
    # !!! set the signal in the solver, but first make a copy !!!
    # this ensures ham_solver is not being modified by this function
    solver_copy = ham_solver.copy()
    solver_copy.signals = [signal]
    
    # call solve
    # !!! use a jax-compatible solver !!!
    results = solver_copy.solve(t_span=[0, dt * len(samples)], y0=np.eye(2, dtype=complex), 
                               method='jax_odeint', atol=1e-8, rtol=1e-8)
    
    return results.y[-1]

We can now `jit` this function.

In [11]:
from jax import jit
jit_simulation_function = jit(simulation_function)

The first time the function is called, it will compile and then execute. Hence, the time taken  on the first call *includes* compilation time.

In [12]:
# block_until_ready is used to make sure time is correctly measured
%time jit_simulation_function(np.ones(100)).block_until_ready()

CPU times: user 647 ms, sys: 10.6 ms, total: 658 ms
Wall time: 656 ms


DeviceArray([[-1.00000012e+00+4.28459274e-11j,
               7.45354887e-09-3.79530561e-07j],
             [-7.45354888e-09-3.79530561e-07j,
              -1.00000012e+00-4.28459249e-11j]], dtype=complex128)

The second time it is called, the compiled function is executed, and therefore demonstrates the true time of the compiled function.

In [13]:
%time jit_simulation_function(2 * np.ones(100)).block_until_ready()

CPU times: user 17 ms, sys: 1.32 ms, total: 18.4 ms
Wall time: 16.5 ms


DeviceArray([[ 1.00000018e+00-3.19374866e-09j,
              -2.30122175e-09+3.18206246e-06j],
             [ 2.30122174e-09+3.18206246e-06j,
               1.00000018e+00+3.19374866e-09j]], dtype=complex128)

### Note:

JAX compiles functions for *fixed* array input shapes and types. Hence, if a different array input shape is supplied, the function will recompile. In this example, this means that if we supply a list of samples of a different length, the function will recompile.

In [14]:
%time jit_simulation_function(np.ones(99)).block_until_ready()

CPU times: user 620 ms, sys: 7.26 ms, total: 628 ms
Wall time: 626 ms


DeviceArray([[-9.99506666e-01+1.57041797e-05j,
               5.54722383e-10-3.14110313e-02j],
             [-5.54722386e-10-3.14110313e-02j,
              -9.99506666e-01-1.57041797e-05j]], dtype=complex128)